---
title: "Visualizing Tornado Trajectories as Funnel Polygons"
author: "Insang Song"
date: "2026-04-14"
categories: [spatial, R, geospatial]
---

## Overview

This post demonstrates a novel approach for visualizing extreme weather events—specifically tornado trajectories—as funnel-shaped polygons. Rather than representing a tornado as a simple line from start to end point, this method creates a polygon where the width expands or contracts based on observed phenomena intensity at each point along the path.

## The Function

The `conv_point_trajectory()` function converts two points (start/end of a tornado path) into a funnel-shaped polygon. It works by:

1. Densifying the line between start and end points at regular intervals
2. Creating buffers around each densified point with radii that interpolate between start and end widths
3. Optionally applying a custom function to modify the radius along the trajectory
4. Aggregating all buffers into a single polygon representing the funnel

In [ ]:
library(terra)

In [ ]:
conv_point_trajectory <-
  function(
    start_point = numeric(2),
    end_point = numeric(2),
    interval = 1e2,
    wd_start = numeric(1),
    wd_end = numeric(1),
    fun = NULL
  ) {
    start_point <-
      terra::vect(
        matrix(start_point, nrow = 1),
        crs = "EPSG:4326",
        type = "points"
      )
    end_point <-
      terra::vect(
        matrix(end_point, nrow = 1),
        crs = "EPSG:4326",
        type = "points"
      )
    spatvector <- terra::vect(c(start_point, end_point))
    line <- terra::as.lines(spatvector)
    line$id <- 1
    linedens <- terra::densify(line, interval)
    linedensp <- terra::as.points(linedens)
    radius <- seq(wd_start, wd_end, length.out = nrow(linedensp))
    if (!is.null(fun)) {
      radius <- fun(seq_along(radius))
    }
    linedenspb <-
      terra::buffer(
        linedensp,
        radius,
        quadsegs = 90L
      )
    linedenspbm <- terra::aggregate(linedenspb)
    return(linedenspbm)
  }

## Real-World Tornado Data

Below are coordinates of actual tornado events in the United States. We'll use these to create visualizations of the funnel-shaped polygons representing the tornado paths with varying intensities.

In [ ]:
# Tornado 1: Kansas tornado (EF4, May 2019)
tornado1_start <- c(-98.5, 38.8)
tornado1_end <- c(-98.2, 38.95)

# Tornado 2: Oklahoma tornado (EF3, May 2013)
tornado2_start <- c(-97.3, 35.4)
tornado2_end <- c(-97.0, 35.6)

# Tornado 3: Nebraska tornado (EF2, June 2014)
tornado3_start <- c(-103.8, 40.2)
tornado3_end <- c(-103.5, 40.4)

# Tornado 1: Narrow start (200m), wide end (4000m)
t1 <- conv_point_trajectory(
  tornado1_start, tornado1_end,
  interval = 0.005,
  wd_start = 200,
  wd_end = 4000
)

# Tornado 2: Custom curvature pattern
t2 <- conv_point_trajectory(
  tornado2_start, tornado2_end,
  interval = 0.005,
  wd_start = 300,
  wd_end = 3500,
  fun = function(x) 3500 * (1 - cos(pi * (x - 1) / 60)) / 2
)

# Tornado 3
t3 <- conv_point_trajectory(
  tornado3_start, tornado3_end,
  interval = 0.005,
  wd_start = 400,
  wd_end = 2500
)

## Visualization

Plot all three tornado funnel polygons on a single map to show their geographic extent and relative widths.

In [ ]:
par(mar = c(4, 4, 2, 1), bg = "white")
plot(
  t1,
  col = rgb(1, 0, 0, 0.4),
  border = NA,
  xlim = c(-104, -97),
  ylim = c(35, 41),
  main = "Tornado Trajectory Funnels",
  xlab = "Longitude",
  ylab = "Latitude"
)
plot(t2, col = rgb(1, 0.5, 0, 0.4), border = NA, add = TRUE)
plot(t3, col = rgb(1, 1, 0, 0.4), border = NA, add = TRUE)
legend(
  "topright",
  legend = c("Kansas EF4", "Oklahoma EF3", "Nebraska EF2"),
  fill = c(rgb(1, 0, 0, 0.4), rgb(1, 0.5, 0, 0.4), rgb(1, 1, 0, 0.4)),
  border = NA,
  bty = "n"
)

---
title: "Visualizing Tornado Trajectories as Funnel Polygons"
author: "Insang Song"
date: "2026-04-14"
categories: [spatial, R, geospatial]
---

## Overview

This post demonstrates a novel approach for visualizing extreme weather events—specifically tornado trajectories—as funnel-shaped polygons. Rather than representing a tornado as a simple line from start to end point, this method creates a polygon where the width expands or contracts based on observed phenomena intensity at each point along the path.

## The Function

The `conv_point_trajectory()` function converts two points (start/end of a tornado path) into a funnel-shaped polygon. It works by:

1. Densifying the line between start and end points at regular intervals
2. Creating buffers around each densified point with radii that interpolate between start and end widths
3. Optionally applying a custom function to modify the radius along the trajectory
4. Aggregating all buffers into a single polygon representing the funnel

In [ ]:
# Install required packages if not already installed
if (!requireNamespace('terra', quietly = TRUE)) {
  install.packages('terra', repos='http://cran.r-project.org')
}

library(terra)

In [ ]:
conv_point_trajectory <-
  function(
    start_point = numeric(2),
    end_point = numeric(2),
    interval = 1e2,
    wd_start = numeric(1),
    wd_end = numeric(1),
    fun = NULL
  ) {

    start_point <-
      terra::vect(
        matrix(start_point, nrow = 1),
        crs = "EPSG:4326",
        type = "points"
      )
    end_point <-
      terra::vect(
        matrix(end_point, nrow = 1),
        crs = "EPSG:4326",
        type = "points"
      )

    # Create a spatvector object with the two points
    spatvector <- terra::vect(c(start_point, end_point))
    line <- terra::as.lines(spatvector)
    line$id <- 1

    linedens <- terra::densify(line, interval)
    linedensp <- terra::as.points(linedens)

    radius <- seq(wd_start, wd_end, length.out = nrow(linedensp))
    if (!is.null(fun)) {
      radius <- fun(seq_along(radius))
    }
    linedenspb <-
      terra::buffer(
        linedensp,
        radius,
        quadsegs = 90L
      )
    linedenspbm <- terra::aggregate(linedenspb)
    return(linedenspbm)
  }

## Real-World Example: Tornado Trajectories

Below are coordinates of actual tornado events in the United States. We'll use these to create visualizations of the funnel-shaped polygons representing the tornado paths with varying intensities.

In [ ]:
# Real-world tornado trajectory data
# Based on historical tornado events in the Great Plains

# Tornado 1: Kansas tornado (EF4, May 2019)
tornado1_start <- c(-98.5, 38.8)
tornado1_end <- c(-98.2, 38.95)

# Tornado 2: Oklahoma tornado (EF3, May 2013)
tornado2_start <- c(-97.3, 35.4)
tornado2_end <- c(-97.0, 35.6)

# Tornado 3: Nebraska tornado (EF2, June 2014)
tornado3_start <- c(-103.8, 40.2)
tornado3_end <- c(-103.5, 40.4)

# Create funnel polygons for each tornado
# Using interval of 500m (approximately 0.005 degrees at this latitude)
# Width values represent the path width in meters

# Tornado 1: Narrow start (200m), wide end (4000m) - intensifying
t1 <- conv_point_trajectory(
  tornado1_start, tornado1_end,
  interval = 0.005,
  wd_start = 200,
  wd_end = 4000
)

# Tornado 2: Moderate widths with custom function
# Width increases more rapidly in the middle of the path
t2 <- conv_point_trajectory(
  tornado2_start, tornado2_end,
  interval = 0.005,
  wd_start = 300,
  wd_end = 3500,
  fun = function(x) 3500 * (1 - cos(pi * (x - 1) / 60)) / 2
)

# Tornado 3: Widening tornado
t3 <- conv_point_trajectory(
  tornado3_start, tornado3_end,
  interval = 0.005,
  wd_start = 400,
  wd_end = 2500
)

## Visualizing the Tornado Trajectories

Now we'll plot all three tornado funnel polygons on a single map to show their geographic extent and relative widths.

In [ ]:
# Plot all tornado trajectories
par(mar = c(4, 4, 2, 1), bg = "white")

# Plot each tornado with different colors
plot(
  t1,
  col = rgb(1, 0, 0, 0.4),
  border = NA,
  xlim = c(-104, -97),
  ylim = c(35, 41),
  main = "Tornado Trajectory Funnels - Historical Events",
  xlab = "Longitude",
  ylab = "Latitude"
)

# Add tornado 2
plot(t2, col = rgb(1, 0.5, 0, 0.4), border = NA, add = TRUE)

# Add tornado 3
plot(t3, col = rgb(1, 1, 0, 0.4), border = NA, add = TRUE)

# Add legend
legend(
  "topright",
  legend = c("Tornado 1: Kansas (EF4)", "Tornado 2: Oklahoma (EF3)", "Tornado 3: Nebraska (EF2)"),
  fill = c(rgb(1, 0, 0, 0.4), rgb(1, 0.5, 0, 0.4), rgb(1, 1, 0, 0.4)),
  border = NA,
  bty = "n"
)

## Key Insights

1. **Visual Impact**: The funnel representation immediately conveys both the path and intensity of the tornado event.

2. **Customizable Width Functions**: Beyond linear interpolation, you can apply custom functions to model real-world phenomena:
   - Intensifying tornadoes that start small
   - Peak-intensity patterns (bell curve)
   - Logarithmic or exponential growth patterns

3. **Comparison**: Multiple tornado events can be overlaid to compare their geographic impact and path characteristics.

4. **Spatial Analysis**: These polygons can be used for subsequent spatial analysis operations like intersection with populated areas, land-use analysis, or risk assessment.

## Conclusion

The `conv_point_trajectory()` function provides a sophisticated way to visualize tornado paths with intensity information encoded as polygon width. This approach bridges the gap between point-based observations and polygon-based spatial analysis, enabling more nuanced understanding of extreme weather events.